We will verify complete end to end pipeline

In [0]:
--Validate Bronze record count

SELECT
    COUNT(*) AS Bronze_Member_Records
FROM delta.`abfss://bronze@incubytedatalake01.dfs.core.windows.net/member_data/`;

Bronze_Member_Records
9


In [0]:
--Validate Silver Record Count

SELECT
    COUNT(*) AS Silver_Member_Records
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`;

Silver_Member_Records
9


In [0]:
--Validate Gold country-wise table
SELECT 'Australia' AS Country, COUNT(*) AS Total_Records
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_Australia/`

UNION ALL

SELECT 'India', COUNT(*)
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_India/`

UNION ALL

SELECT 'USA', COUNT(*)
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_USA/`;

Country,Total_Records
Australia,3
India,3
USA,2


In [0]:
--Gold Redemption Query

SELECT
    COUNT(*) AS Total_Redemption_Transactions,
    SUM(Miles_Redeemed) AS Total_Miles_Redeemed,
    COUNT(DISTINCT Transaction_ID) AS Unique_Transactions
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/`;

Total_Redemption_Transactions,Total_Miles_Redeemed,Unique_Transactions
2,17000,2


In [0]:
--Recreate the member-redemption join
CREATE OR REPLACE TEMP VIEW member_redemption_join AS

SELECT
    r.member_id,
    r.Transaction_ID,
    r.Transaction_Date,
    r.Partner,
    r.Miles_Redeemed,
    r.Transaction_Status,
    m.Member_Name,
    m.Source_Country

FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/` r

LEFT JOIN delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/` m

ON r.member_id = CAST(m.Member_ID AS STRING);

In [0]:
--Verify matched and unmatched record
SELECT
    COUNT(*) AS Total_Transactions,
    SUM(CASE WHEN Member_Name IS NOT NULL THEN 1 ELSE 0 END)
        AS Matched_Transactions,
    SUM(CASE WHEN Member_Name IS NULL THEN 1 ELSE 0 END)
        AS Unmatched_Transactions
FROM member_redemption_join;

Total_Transactions,Matched_Transactions,Unmatched_Transactions
2,0,2


In [0]:
--Create the Gold joined table
CREATE OR REPLACE TABLE
delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/member_redemption_join/`
USING DELTA
AS
SELECT
    r.member_id,
    r.Transaction_ID,
    r.Transaction_Date,
    r.Partner,
    r.Miles_Redeemed,
    r.Transaction_Status,
    m.Member_Name,
    m.Source_Country
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/` r
LEFT JOIN delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/` m
    ON r.member_id = CAST(m.Member_ID AS STRING);

num_affected_rows,num_inserted_rows


In [0]:
--Verify joined table
SELECT *
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/member_redemption_join/`;


member_id,Transaction_ID,Transaction_Date,Partner,Miles_Redeemed,Transaction_Status,Member_Name,Source_Country
223457,RX10091,20240110,AeroLink,12000,COMPLETED,null,null
223457,RX10092,20240113,SkyPoints,5000,PENDING,null,null


In [0]:
--Validate Age and Stale_Member
SELECT
    Member_Name,
    Member_ID,
    DOB,
    Age,
    Last_Flight_Date,
    Stale_Member
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
ORDER BY Source_Country, Member_ID;

Member_Name,Member_ID,DOB,Age,Last_Flight_Date,Stale_Member
Mike,1,null,null,2022-08-01,true
Jonnathan,2,1997-12-13,28,2022-01-05,true
Cristina,3,1998-03-12,28,2022-03-20,true
Vikas,1,1998-12-01,27,2022-06-15,true
Rahul,2,1982-08-13,44,2022-03-10,true
Sameer,3,1952-08-13,74,2022-02-25,true
Sam,1,null,null,2022-08-20,true
John,2,null,null,2022-01-15,true
Mike,3,null,null,2021-12-30,true


In [0]:
--Final Pipeline COunt Validation

SELECT 'Bronze Members' AS Layer, COUNT(*) AS Total_Records
FROM delta.`abfss://bronze@incubytedatalake01.dfs.core.windows.net/member_data/`

UNION ALL

SELECT 'Silver Members', COUNT(*)
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`

UNION ALL

SELECT 'Gold Australia', COUNT(*)
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_Australia/`

UNION ALL

SELECT 'Gold India', COUNT(*)
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_India/`

UNION ALL

SELECT 'Gold USA', COUNT(*)
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/Table_USA/`

UNION ALL

SELECT 'Gold Redemptions', COUNT(*)
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/`

UNION ALL

SELECT 'Gold Member Redemption Join', COUNT(*)
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/member_redemption_join/`;

Layer,Total_Records
Bronze Members,9
Silver Members,9
Gold Australia,3
Gold India,3
Gold USA,2
Gold Redemptions,2
Gold Member Redemption Join,2
